# Homework 2 (Fall 2026): Visualizing features from local linearization

We study derivatives of the scalar network output with respect to its parameters, not derivatives of the training loss. Use the supplied widths and training settings. Compare initialization and the final trained network, and use `X_test` for feature plots. Submit the completed notebook's PDF export with your answers and plots.

This notebook is self-contained and generates its own data; it does not clone another course repository. In Google Colab, run the dependency cell below in the current notebook kernel. Locally, follow the adjacent `README.md` and skip that cell after installing the dependencies. Complete the marked TODOs as you reach parts (a), (b), and (c).


In [ ]:
# Install into the current notebook kernel (Colab or a fresh Jupyter kernel).
%pip install "numpy>=1.22,<2" "matplotlib>=3.5,<4" "torch>=2.0,<3" "ipywidgets>=8,<9"


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import copy
import time
from ipywidgets import fixed, interactive, widgets
%matplotlib inline

# Reproducible initialization and minibatch sampling within the chosen device.
np.random.seed(0)
torch.manual_seed(0)


In [ ]:
def to_torch(x):
    return torch.from_numpy(x).float()


def to_numpy(x):
    return x.detach().cpu().numpy()


def plot_data(X, y, X_test, y_test):
    clip_bound = 2.5
    plt.xlim(0, 1)
    plt.ylim(-clip_bound, clip_bound)
    plt.scatter(X[:, 0], y, c='darkorange', s=40.0, label='training data points')
    plt.plot(X_test, y_test, '--', color='royalblue', linewidth=2.0, label='Ground truth')


def plot_relu(bias, slope):
    # Draw the actual ReLU, including elbows that move outside the input range.
    plot_x = np.array([0.0, 1.0])
    if slope != 0:
        elbow = -bias / slope
        if 0 <= elbow <= 1:
            plt.scatter([elbow], [0], c='darkgrey', s=40.0)
            plot_x = np.array([0.0, elbow, 1.0])
    plt.plot(plot_x, np.maximum(slope * plot_x + bias, 0), ':')


def plot_relus(params):
    slopes = to_numpy(params[0]).ravel()
    biases = to_numpy(params[1])
    for relu in range(biases.size):
        plot_relu(biases[relu], slopes[relu])


def plot_function(X_test, net):
    parameter = next(net.parameters())
    inputs = torch.as_tensor(X_test, dtype=parameter.dtype, device=parameter.device)
    with torch.no_grad():
        y_pred = net(inputs)
    plt.plot(X_test, to_numpy(y_pred), '-', color='forestgreen', label='prediction')


def plot_update(X, y, X_test, y_test, net, state=None):
    if state is not None:
        net = copy.deepcopy(net)
        net.load_state_dict(state)
    plt.figure(figsize=(10, 7))
    plot_relus(list(net.parameters()))
    plot_function(X_test, net)
    plot_data(X, y, X_test, y_test)
    plt.legend()
    plt.show();


def train_network(X, y, X_test, y_test, net, optim, n_steps, save_every,
                  initial_weights=None, verbose=False):
    """Train with minibatches of one fifth of the data and preserve checkpoints."""
    if initial_weights is not None:
        net.load_state_dict(initial_weights)
    parameter = next(net.parameters())
    X_train = torch.as_tensor(X, dtype=parameter.dtype, device=parameter.device)
    y_train = torch.as_tensor(y.reshape(-1, 1), dtype=parameter.dtype, device=parameter.device)
    test_inputs = torch.as_tensor(X_test, dtype=parameter.dtype, device=parameter.device)
    test_targets = torch.as_tensor(y_test.reshape(-1, 1), dtype=parameter.dtype, device=parameter.device)
    num_samples = len(X)
    batch_size = max(1, num_samples // 5)
    criterion = nn.MSELoss()
    history = {}
    net.train()
    for step in range(n_steps):
        subsample = torch.randperm(num_samples, device=parameter.device)[:batch_size]
        optim.zero_grad()
        step_loss = criterion(net(X_train[subsample]), y_train[subsample])
        step_loss.backward()
        optim.step()
        if (step + 1) % save_every == 0 or step == 0 or step + 1 == n_steps:
            with torch.no_grad():
                test_loss = criterion(net(test_inputs), test_targets)
            history[step + 1] = {
                'state': copy.deepcopy(net.state_dict()),
                'train_error': step_loss.item(),
                'test_error': test_loss.item()
            }
            if verbose:
                print('SGD Iteration %d' % (step + 1))
                print('\tTrain Loss: %.3f' % step_loss.item())
                print('\tTest Loss: %.3f' % test_loss.item())
            elif (step + 1) % (save_every * 10) == 0:
                print('SGD Iteration %d' % (step + 1))
    return history


def plot_test_train_errors(history):
    plt.figure()
    sample_points = np.array(list(history.keys()))
    etrain = [history[s]['train_error'] for s in history]
    etest = [history[s]['test_error'] for s in history]
    plt.plot(sample_points / 1e3, etrain, label='Train Error')
    plt.plot(sample_points / 1e3, etest, label='Test Error')
    plt.xlabel("Iterations (1000's)")
    plt.ylabel("MSE")
    plt.yscale('log')
    plt.legend()
    plt.show();


def make_iter_slider(iters):
    return widgets.SelectionSlider(
        options=iters,
        value=iters[0],
        description='SGD Iterations: ',
        disabled=False
    )


def history_interactive(history, idx, X, y, X_test, y_test, net):
    plot_update(X, y, X_test, y_test, net, state=history[idx]['state'])
    print("Train Error: %.3f" % history[idx]['train_error'])
    print("Test Error: %.3f" % history[idx]['test_error'])


def make_history_interactive(history, X, y, X_test, y_test, net):
    sample_points = list(history.keys())
    return interactive(history_interactive,
                       history=fixed(history),
                       idx=make_iter_slider(sample_points),
                       X=fixed(X),
                       y=fixed(y),
                       X_test=fixed(X_test),
                       y_test=fixed(y_test),
                       net=fixed(net))


%matplotlib inline

# Generate Training and Test Data

We are using piecewise linear function. Our training data has added noise $y = f(x) + \epsilon,\, \epsilon \sim \mathcal{N}(0, \sigma^2)$. The test data is noise free.

_Once you have gone through the discussion once you may wish to adjust the number of training samples and noise variance to see how gradient descent behaves under the new conditions._

In [ ]:
f_type = 'piecewise_linear'

def f_true(X, f_type):
    if f_type == 'sin(20x)':
        return np.sin(20 * X[:,0])
    else:
        TenX = 10 * X[:,0]
        _ = 12345
        return (TenX - np.floor(TenX)) * np.sin(_ * np.ceil(TenX)) - (TenX - np.ceil(TenX)) * np.sin(_ * np.floor(TenX))

n_features = 1
n_samples = 200
sigma = 0.1
rng = np.random.RandomState(1)

# Generate train data
X = np.sort(rng.rand(n_samples, n_features), axis=0)
y = f_true(X, f_type) + rng.randn(n_samples) * sigma

# Generate NOISELESS test data
X_test = np.concatenate([X.copy(), np.expand_dims(np.linspace(0., 1., 1000), axis=1)])
X_test = np.sort(X_test, axis=0)
y_test = f_true(X_test, f_type)

In [ ]:
plt.scatter(X, y)
plt.show()

In [ ]:
plt.scatter(X_test, y_test)
plt.show()

# Define the neural networks

We fit the scalar target with a network containing one ReLU hidden layer:
$$\hat y=W^{(2)}\operatorname{ReLU}(W^{(1)}x+b^{(1)})+b^{(2)}.$$

Use hidden widths 10, 20, and 40. Initial biases place the first-layer ReLU elbows inside $[0,1]$. The next section trains all parameters with SGD at learning rate 0.02 and keeps the initial and trained networks separately for comparison.


In [ ]:
# Don't rerun this cell after training or you will lose all your work
nets_by_size = {}

In [ ]:
widths = [10, 20, 40]
lr_all = 0.02
for width in widths:
    net = nn.Sequential(nn.Linear(1, width), nn.ReLU(), nn.Linear(width, 1))
    # Place each initial elbow -bias/weight inside [0, 1].
    elbows = np.sort(np.random.rand(width))
    with torch.no_grad():
        net[0].bias.copy_(to_torch(-elbows * to_numpy(net[0].weight).ravel()))
    nets_by_size[width] = {
        'net': net,
        'init': copy.deepcopy(net.state_dict())
    }


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for width, net in nets_by_size.items():
  net['net'].to(device=device)

# Train the neural networks

In [ ]:
n_steps = 150000
save_every = 1000
t0 = time.time()
for w in widths:
    print("-"*40)
    print("Width", w)
    new_net = nn.Sequential(nn.Linear(1, w),
                        nn.ReLU(),
                        nn.Linear(w, 1))
    new_net.load_state_dict(nets_by_size[w]['net'].state_dict().copy())
    new_net.to(device=device)
    opt_all = torch.optim.SGD(params=new_net.parameters(), lr=lr_all)
    initial_weights = nets_by_size[w]['init']
    history_all = train_network(X, y, X_test, y_test,
                            new_net, optim=opt_all,
                            n_steps=n_steps, save_every=save_every,
                            initial_weights=initial_weights,
                            verbose=False)
    nets_by_size[w]['trained_net'] = new_net
    nets_by_size[w]['hist_all'] = history_all
    print("Width", w)
    plot_test_train_errors(history_all)
t1 = time.time()
print("-"*40)
print("Trained all layers in %.1f minutes" % ((t1 - t0) / 60))

# (a) Visualize output-gradient features

Write the one-hidden-layer network as
$$y(x;\theta)=\sum_i a_i\operatorname{ReLU}(w_i x+b_i)+c.$$
At a fixed parameter snapshot, the local feature for parameter $\theta_j$ is
$\phi_j(x)=\partial y(x;\theta)/\partial\theta_j$.

Complete the two backpropagation lines below. Plot the features for the first hidden layer's weights and biases separately, using `X_test` as the input grid, at initialization and after training. Clear gradients between inputs and differentiate the scalar output. Use PyTorch's derivative convention at ReLU kinks.


In [ ]:
def backward_and_plot_grad(X, model, vis_name='all', title='', legend=False):
    """Plot output derivatives for the selected parameter group at fixed weights."""
    parameter = next(model.parameters())
    gradient_collect = {}
    for x in X:
        x_tensor = torch.as_tensor(x, dtype=parameter.dtype, device=parameter.device)
        y = model(x_tensor)

        ########################################################################
        # TODO: Clear parameter gradients, then backpropagate the scalar y.
        # (2 lines)
        ########################################################################
        # (YOUR CODE HERE)
        ########################################################################

        for name, param in model.named_parameters():
            if vis_name == 'all' or vis_name == name:
                for index, gradient in enumerate(param.grad.detach().reshape(-1).cpu().numpy()):
                    key = f'{name}.{index}'
                    gradient_collect.setdefault(key, []).append(gradient.item())

    plt.figure()
    for name, values in gradient_collect.items():
        plt.plot(X, values, label=name)
    plt.xlabel('Input x')
    plt.ylabel(f'Output derivative: {vis_name}')
    if legend:
        plt.legend()
    plt.title(title)
    plt.show()


for width in nets_by_size:
    for key, label in [('net', 'Initialization'), ('trained_net', 'After training')]:
        model = nets_by_size[width][key]
        for group in ['0.weight', '0.bias']:
            backward_and_plot_grad(X_test, model, group, f'{label}; width {width}')


# (b) SVD of the feature matrix

Collect **all weights and biases, including the output layer**, in a fixed parameter order. For each network snapshot, form $\Phi_{ij}=\phi_j(x_i)$ using the $n$ training inputs `X`. If there are $d$ parameters and $r=\min(n,d)$, the thin SVD has
$$\Phi=U\Sigma V^\top,\qquad U\in\mathbb R^{n\times r},\quad \Sigma\in\mathbb R^{r\times r},\quad V\in\mathbb R^{d\times r}.$$

Plot all singular values and the leading ten principal features (or all if $r<10$), both at initialization and after training. The $k$th principal feature is $\psi_k(x)=\sum_j V_{jk}\phi_j(x)$.

For plots between training points, evaluate the same output derivatives at `X_test` to obtain $\Phi_{\rm plot}$, then compute $\Phi_{\rm plot}V$. Use the same network snapshot and parameter order, and **do not fit another SVD on `X_test`**. Singular-vector signs are arbitrary. The helper below collects features at either set of inputs; complete its backpropagation lines, the SVD, and the principal-feature projection.


In [ ]:
def collect_output_features(inputs, model):
    """Return one row per input and one column per weight/bias parameter."""
    parameter = next(model.parameters())
    parameters = tuple(model.parameters())  # same order for train and plot inputs
    rows = []
    for x in inputs:
        x_tensor = torch.as_tensor(x, dtype=parameter.dtype, device=parameter.device)
        y = model(x_tensor)

        ########################################################################
        # TODO: Clear gradients and backpropagate the scalar output (as in a).
        # (2 lines)
        ########################################################################
        # (YOUR CODE HERE)
        ########################################################################

        rows.append(np.concatenate([
            param.grad.detach().reshape(-1).cpu().numpy()
            for param in parameters
        ]))
    return np.stack(rows)


def compute_svd_plot_features(X, y, X_test, y_test, model, title=''):
    # Labels y and y_test are not used: these are output derivatives, not losses.
    feature_matrix = collect_output_features(X, model)
    n, d = feature_matrix.shape
    r = min(n, d)

    ############################################################################
    # TODO: Compute a thin SVD (1 line). Shapes: u=(n,r), s=(r,), vh=(r,d).
    ############################################################################
    # (YOUR CODE HERE)
    ############################################################################

    # Evaluate at plotting inputs using the same model and parameter ordering.
    plot_feature_matrix = collect_output_features(X_test, model)
    ############################################################################
    # TODO: Project the plotting features onto the SAME right singular vectors.
    # Store the result in principal_features (1 line); do not refit the SVD.
    ############################################################################
    # (YOUR CODE HERE)
    ############################################################################

    plt.figure()
    plt.scatter(np.arange(1, len(s) + 1), s, c='darkorange', s=20)
    plt.xlabel('Singular-value index')
    plt.ylabel('Singular value')
    plt.title(title)
    plt.show()

    plt.figure()
    for index in range(min(10, r)):
        plt.plot(X_test, principal_features[:, index], label=f'Feature {index + 1}')
    plt.xlabel('Input x')
    plt.ylabel('Principal feature')
    plt.title(title)
    plt.legend()
    plt.show()
    return s, principal_features


for width in widths:
    for key, label in [('net', 'Initialization'), ('trained_net', 'After training')]:
        compute_svd_plot_features(
            X, y, X_test, y_test, nets_by_size[width][key],
            title=f'{label}; width {width}'
        )


# (c) Two-hidden-layer network

Add a second fully connected hidden layer of the same width as the first. Use ReLU after each hidden layer and a linear scalar output: `1 -> width -> width -> 1`. Keep the supplied widths, optimizer, learning rate, and training budget.

Repeat parts (a) and (b) at initialization and after training. Plot individual output-gradient features separately for each hidden-layer parameter group (`0.weight`, `0.bias`, `2.weight`, and `2.bias`). Compute one combined thin SVD using **all** weights and biases, including the output layer. Reuse the feature-collection and plotting helpers; their parameter ordering and thin-SVD shapes also work when there are more parameters than training inputs. Keep each model and its inputs on the same device.


In [ ]:
################################################################################
# TODO: Define and train the two-hidden-layer networks for widths [10, 20, 40].
# Keep a separate initial and trained model for each width, and retain the
# supplied SGD settings (lr_all, n_steps, and save_every).
# Reuse backward_and_plot_grad with X_test for each hidden-layer parameter group.
# Reuse compute_svd_plot_features for one combined all-parameter SVD, at both
# initialization and after training. Its feature helper uses each model's device.
################################################################################
# (YOUR CODE HERE)
################################################################################
